#  주요 언어 모델 공급자 및 활용 

- 주요 LLM 공급자(Google Gemini, Groq, Ollama)의 특징과 차이점을 이해한다
- LangChain을 활용하여 다양한 LLM을 RAG 체인에 통합하는 방법을 습득한다
- 각 공급자의 모델을 실제로 사용해보고 응답 품질과 성능을 비교한다

---

## 사전 준비

### 필수 패키지
- `langchain-openai`
- `langchain-google-genai`
- `langchain-groq`
- `langchain-ollama`
- `langchain-chroma`
- `langfuse`

### API 키 설정
다음 환경 변수를 `.env` 파일에 설정해야 합니다:
- `OPENAI_API_KEY`: OpenAI API 키
- `GOOGLE_API_KEY`: Google AI API 키
- `GROQ_API_KEY`: Groq API 키
- `LANGFUSE_PUBLIC_KEY`: Langfuse 공개 키
- `LANGFUSE_SECRET_KEY`: Langfuse 비밀 키

### Ollama 설치
Ollama를 사용하려면 로컬 환경에 Ollama를 먼저 설치해야 합니다:
- 공식 웹사이트: https://ollama.com/
- 설치 후 `ollama pull gemma3:latest` 명령으로 모델 다운로드

## 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

import uuid

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

`(3) langfuse handler 설정`

In [3]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

`(4) 벡터스토어 로드`

In [4]:
# 벡터 저장소 로드 
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

print(chroma_db._collection.count())

39


`(5) 벡터 검색기 생성`

In [ ]:
# 기본 retriever 초기화
chroma_k_retriever = chroma_db.as_retriever(
    search_kwargs={"k": 4}
)

query = "테슬라의 자율주행 기술의 특징은 무엇인가요?"
retrieved_docs = chroma_k_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

TypeError: Collection.query() got an unexpected keyword argument 'callbacks'

`(6) RAG 체인 생성`

In [6]:
# 각 쿼리에 대한 검색 결과를 한꺼번에 Context로 전달해서 답변을 생성
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

def create_rag_chain(retriever, llm):

    template = """Answer the following question based on this context. If the context is not relevant to the question, just answer with '답변에 필요한 근거를 찾지 못했습니다.'

    [Context]
    {context}

    [Question]
    {question}

    [Answer]
    """

    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join([doc.page_content for doc in docs])

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()} 
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

In [13]:
# RAG 체인 생성 및 테스트 (OpenAI의 gpt-4.1-nano 사용)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")
openai_rag_chain = create_rag_chain(chroma_k_retriever, llm)

query = "테슬라의 자율주행 기술의 특징은 무엇인가요?"
answer = openai_rag_chain.invoke(
    query,
    config={"callbacks": [langfuse_handler]}
)

print(answer)

Tesla의 자율주행 기술인 Autopilot은 고급 운전자 지원 시스템(ADAS)으로, 부분적인 차량 자동화를 의미합니다.


---

## 주요 **LLM 모델 공급자** 

- **LLM 공급자**는 대규모 언어 모델을 개발하고 서비스를 제공하는 기업들을 의미함
- **기술 서비스** 제공 방식에 따라 다양한 기업들이 시장에 참여하고 있음


### 1) **Google Gemini** API

- **Google의 최신 AI 모델** Gemini가 API를 통해 개발자들에게 공개됨
- API는 **세 가지 모델 버전**을 제공: Gemini Pro, Gemini Pro Vision, Ultra 모델
- **텍스트, 이미지, 오디오** 등 다양한 형식의 입력을 처리할 수 있는 **멀티모달 기능** 지원
- 무료 체험판과 함께 **사용량 기반 과금 체계** 도입

- 참고: https://ai.google.dev/
- 설치: `pip install langchain_google_genai` / ```uv add langchain_google_genai```

- **API 키 설정**
- 환경 변수: **GOOGLE_API_KEY**

In [10]:
# ChatGoogleGenerativeAI - LLM 모델 
from langchain_google_genai import ChatGoogleGenerativeAI

# 모델 로드 
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

# RAG 체인 생성 및 테스트
google_genai_rag_chain = create_rag_chain(chroma_k_retriever, llm)

query = "테슬라의 자율주행 기술의 특징은 무엇인가요?"
answer = google_genai_rag_chain.invoke(
    query,
    config={"callbacks": [langfuse_handler]}
)

print(answer)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

### **[실습]**

- Google Gemini에서 제공하는 다른 모델을 RAG 체인에 적용합니다. 
- 각 모델이 생성한 답변의 품질과 특성을 비교합니다. 
- 모델 확인: https://ai.google.dev/gemini-api/docs

In [ ]:
# 여기에 코드를 작성하세요.

### 2) **Groq** API

- **초고속 추론 속도**를 제공하는 새로운 LLM API 서비스 출시
- **오픈소스 모델** 지원으로 Llama, Mixtral 등 다양한 기존 모델 활용 가능
- **1초 미만의 응답 시간**을 목표로 하는 **LPU**(Language Processing Unit) 기술 도입
- API 사용량에 따른 **합리적인 가격 정책** 제시

- 참고: https://groq.com/
- 설치: `pip install langchain_groq` / ```uv add langchain_groq```

- **API 키 설정**
- 환경 변수: **GROQ_API_KEY**

In [8]:
# 환경변수 로드
load_dotenv(override=True)

True

In [9]:
# ChatGroq - LLM 모델 
from langchain_groq import ChatGroq

# 모델 로드 
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.0,
    max_tokens=500,
)

# RAG 체인 생성 및 테스트
groq_rag_chain = create_rag_chain(chroma_k_retriever, llm)

query = "테슬라의 자율주행 기술의 특징은 무엇인가요?"
answer = groq_rag_chain.invoke(
    query,
    config={"callbacks": [langfuse_handler]}
)

print(answer)

Tesla의 자율주행 기술은 **Autopilot**이라는 고급 운전자 지원 시스템(ADAS)으로 구현됩니다. Autopilot은 Tesla가 자체 개발한 기술이며, 차량의 **부분적인 자동화**를 제공하여 운전자가 일정 상황에서 조작을 보조받을 수 있게 합니다. 즉, 완전한 자율주행이 아니라 운전 보조 수준의 기능을 중심으로 설계되어 있습니다.


### **[실습]**

- Groq에서 제공하는 다른 모델을 RAG 체인에 적용합니다. 
- 각 모델이 생성한 답변의 품질과 특성을 비교합니다. 
- 모델 확인: https://console.groq.com/docs/models

In [ ]:
# 여기에 코드를 작성하세요.

### 3) **Ollama** API

- **로컬 환경**에서 LLM을 쉽게 실행할 수 있는 **오픈소스 플랫폼**
- **다양한 LLM 모델** 지원 (Llama, Mistral, CodeLlama 등)
- **단순한 명령어**로 모델 다운로드 및 실행 가능
- 개인용 컴퓨터에서 AI 모델을 쉽게 활용할 수 있는 효율적인 솔루션 제공 (**macOS, Linux, Windows** 운영체제 지원)

- 웹사이트: https://ollama.com/
- 깃허브: https://github.com/ollama/ollama
- 설치: `pip install langchain_ollama` / ```uv add langchain_ollama```

In [11]:
# ChatOllama - LLM 모델 
from langchain_ollama import ChatOllama

# 모델 로드 
llm = ChatOllama(
    model = "exaone3.5:7.8b",   # exaone3.5:7.8b 모델 적용 (ollama pull exaone3.5:7.8b)
    temperature = 0,
    num_predict = 200,
)

# RAG 체인 생성 및 테스트
ollama_rag_chain = create_rag_chain(chroma_k_retriever, llm)

query = "테슬라의 자율주행 기술의 특징은 무엇인가요?"
answer = ollama_rag_chain.invoke(
    query,
    config={"callbacks": [langfuse_handler]}
)

print(answer)

테슬라의 자율주행 기술인 **Autopilot**은 고급 운전자 지원 시스템(ADAS)으로, 차량에 부분적인 자동화 기능을 제공합니다. 이 시스템은 운전자의 부담을 줄이고 안전성을 높이기 위해 설계되었지만, 완전한 자율 주행은 아직 구현되지 않았으며, 운전자의 주의와 제어가 여전히 필요합니다. 문서에서는 Autopilot이 **부분적인 차량 자동화**를 의미한다고 명시하고 있습니다.


### **[실습]**

- Ollama에서 제공하는 다른 모델을 RAG 체인에 적용합니다. 
- 각 모델이 생성한 답변의 품질과 특성을 비교합니다. 
- 모델 확인: https://ollama.com/library

In [ ]:
# 여기에 코드를 작성하세요.